# SAM3 (Segment Anything Model 3) — MLX Demo

Open-vocabulary **object detection**, **instance segmentation**, and **video tracking** on Apple Silicon.

Model: [`facebook/sam3`](https://huggingface.co/facebook/sam3) (~860M parameters)

## Setup

In [ ]:
!uv pip install -U mlx-vlm matplotlib

In [ ]:
import numpy as np
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline

COLORS = [
    (0.12, 0.47, 0.71),  # blue
    (1.00, 0.50, 0.05),  # orange
    (0.17, 0.63, 0.17),  # green
    (0.84, 0.15, 0.16),  # red
    (0.58, 0.40, 0.74),  # purple
    (0.55, 0.34, 0.29),  # brown
    (0.89, 0.47, 0.76),  # pink
    (0.09, 0.75, 0.81),  # cyan
]


def fetch_image(url):
    return Image.open(BytesIO(requests.get(url, timeout=15).content)).convert("RGB")


def nms_results(result, iou_thresh=0.5):
    """Remove duplicate detections via NMS."""
    if len(result.scores) == 0:
        return result
    boxes, scores, masks = result.boxes, result.scores, result.masks
    order = np.argsort(-scores)
    keep = []
    for i in order:
        discard = False
        for j in keep:
            x1 = max(boxes[i][0], boxes[j][0])
            y1 = max(boxes[i][1], boxes[j][1])
            x2 = min(boxes[i][2], boxes[j][2])
            y2 = min(boxes[i][3], boxes[j][3])
            inter = max(0, x2 - x1) * max(0, y2 - y1)
            a_i = (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1])
            a_j = (boxes[j][2] - boxes[j][0]) * (boxes[j][3] - boxes[j][1])
            if inter / max(a_i + a_j - inter, 1e-6) > iou_thresh:
                discard = True
                break
        if not discard:
            keep.append(i)
    from mlx_vlm.models.sam3.generate import DetectionResult
    return DetectionResult(boxes=boxes[keep], masks=masks[keep], scores=scores[keep])


def draw_overlay(ax, image, result, title=""):
    """Draw masks + boxes on axes."""
    W, H = image.size
    ax.imshow(image)
    for i in range(len(result.scores)):
        color = COLORS[i % len(COLORS)]
        mask = result.masks[i]
        if mask.shape[0] != H or mask.shape[1] != W:
            mask = np.array(Image.fromarray(mask.astype(np.float32)).resize((W, H), Image.BILINEAR))
        binary = mask > 0
        overlay = np.zeros((H, W, 4))
        for c in range(3):
            overlay[:, :, c] = binary * color[c]
        overlay[:, :, 3] = binary * 0.5
        ax.imshow(overlay)
        x1, y1, x2, y2 = result.boxes[i]
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2.5, edgecolor=color, facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            x1, max(y1 - 8, 12), f"{result.scores[i]:.2f}",
            color="white", fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.85),
        )
    ax.set_title(title, fontsize=14)
    ax.axis("off")

## Load Model

In [ ]:
from mlx_vlm.utils import load_model, get_model_path
from mlx_vlm.models.sam3.generate import Sam3Predictor
from mlx_vlm.models.sam3.processing_sam3 import Sam3Processor

model_path = get_model_path("facebook/sam3")
model = load_model(model_path)
processor = Sam3Processor.from_pretrained(str(model_path))
predictor = Sam3Predictor(model, processor, score_threshold=0.3)

In [ ]:
# Load test image
image = fetch_image("http://images.cocodataset.org/val2017/000000039769.jpg")
image

## 1. Object Detection

Detect objects by text prompt — returns bounding boxes and confidence scores.

In [ ]:
result = predictor.predict(image, text_prompt="a remote")
result = nms_results(result)

print(f'Prompt: "a remote" — {len(result.scores)} detections')
for i in range(len(result.scores)):
    x1, y1, x2, y2 = result.boxes[i]
    print(f"  [{result.scores[i]:.2f}] box=({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(image)
for i in range(len(result.scores)):
    color = COLORS[i % len(COLORS)]
    x1, y1, x2, y2 = result.boxes[i]
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=3, edgecolor=color, facecolor="none",
    )
    ax.add_patch(rect)
    ax.text(
        x1, max(y1 - 8, 12), f"remote {result.scores[i]:.2f}",
        color="white", fontsize=13, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.85),
    )
ax.set_title('Object Detection — "a remote"', fontsize=15)
ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Instance Segmentation

Same call — per-instance masks are returned alongside boxes.

In [ ]:
result = predictor.predict(image, text_prompt="a cat")
result = nms_results(result)

print(f'Prompt: "a cat" — {len(result.scores)} instances')
for i in range(len(result.scores)):
    area = int(result.masks[i].sum())
    print(f"  [{result.scores[i]:.2f}] mask area = {area:,} px")

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
draw_overlay(ax, image, result, title='Instance Segmentation — "a cat"')
plt.tight_layout()
plt.show()

## 3. Overlay Masks on Image


In [ ]:
result = nms_results(predictor.predict(image, text_prompt="a cat"))

overlay = np.array(image).copy()
colors_rgb = [(30, 120, 255), (255, 80, 30), (30, 200, 30)]
W, H = image.size

for i in range(len(result.scores)):
    mask = result.masks[i]
    if mask.shape != (H, W):
        mask = np.array(Image.fromarray(mask.astype(np.float32)).resize((W, H)))
    binary = mask > 0
    color = np.array(colors_rgb[i % len(colors_rgb)])
    overlay[binary] = (overlay[binary] * 0.5 + color * 0.5).astype(np.uint8)

Image.fromarray(overlay)

## 4. Box-Guided Detection

Pass bounding box prompts to guide detection to specific image regions.

In [ ]:
# Box covering the left cat in xyxy pixel coordinates
boxes = np.array([[0, 50, 350, 480]])

result_text = nms_results(predictor.predict(image, text_prompt="a cat"))
result_box = nms_results(predictor.predict(image, text_prompt="a cat", boxes=boxes))

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

draw_overlay(axes[0], image, result_text, title='Text only — "a cat"')

draw_overlay(axes[1], image, result_box, title='Text + box prompt — "a cat"')
x1, y1, x2, y2 = boxes[0]
axes[1].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=3, ec="yellow", fc="none", ls="--"))
axes[1].text(x1, y1 - 8, "input box", color="yellow", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Semantic Segmentation

Access both instance masks and the dense semantic segmentation output.

In [ ]:
import mlx.core as mx

inputs = processor.preprocess_image(image)
text_inputs = processor.preprocess_text("a cat")

outputs = model.detect(
    mx.array(inputs["pixel_values"]),
    mx.array(text_inputs["input_ids"]),
    mx.array(text_inputs["attention_mask"]),
)
mx.eval(outputs)

# Get best instance mask
pred_logits = np.array(outputs["pred_logits"][0])
scores = 1 / (1 + np.exp(-pred_logits))
best = int(np.argmax(scores))
best_mask = np.array(outputs["pred_masks"][0, best])

# Semantic segmentation
semantic = np.array(outputs["semantic_seg"][0, 0])

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
axes[0].imshow(image)
axes[0].set_title("Original", fontsize=14)
axes[0].axis("off")

axes[1].imshow(best_mask, cmap="RdYlBu")
axes[1].set_title(f"Instance Mask (query {best}, score {scores[best]:.2f})", fontsize=14)
axes[1].axis("off")

axes[2].imshow(semantic, cmap="viridis")
axes[2].set_title("Semantic Segmentation", fontsize=14)
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 6. Multi-Prompt Detection

Run different text prompts on different images.

In [ ]:
urls_prompts = [
    ("http://images.cocodataset.org/val2017/000000039769.jpg", "a cat"),
    ("http://images.cocodataset.org/val2017/000000397133.jpg", "an airplane"),
    ("http://images.cocodataset.org/val2017/000000252219.jpg", "a zebra"),
    ("http://images.cocodataset.org/val2017/000000087038.jpg", "a car"),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (url, prompt) in enumerate(urls_prompts):
    print(f'[{idx+1}/4] "{prompt}"...', end=" ", flush=True)
    try:
        img = fetch_image(url)
    except Exception as e:
        print(f"FAILED ({e})")
        axes[idx].text(0.5, 0.5, "Failed to load", ha="center", va="center", fontsize=14)
        axes[idx].set_title(f'"{prompt}"')
        axes[idx].axis("off")
        continue

    res = predictor.predict(img, text_prompt=prompt, score_threshold=0.1)
    res = nms_results(res)
    print(f"{len(res.scores)} detections")
    draw_overlay(axes[idx], img, res, title=f'"{prompt}" ({len(res.scores)} det.)')

plt.suptitle("SAM3 MLX — Multi-Prompt Detection", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Video Tracking

Track objects across video frames. Runs detection per-frame and overlays masks, contours, and bounding boxes.

> Replace the video path below with your own video file.

In [ ]:
import cv2
from pathlib import Path

COLORS_BGR = [
    (181, 120, 31), (13, 128, 255), (43, 161, 43),
    (41, 38, 214), (189, 102, 148), (74, 87, 140),
]

VIDEO_PATH = "./videos/car_video.mp4"  # <-- change this
TEXT_PROMPT = "a car"
PROCESS_EVERY = 2  # run detection every N frames (1 = every frame)

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video: {total_frames} frames, {fps:.1f} fps, {W}x{H}")

# Output video
out_path = "sam3_outputs/4_tracking_video.mp4"
Path("sam3_outputs").mkdir(exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(out_path, fourcc, fps, (W, H))

latest_masks, latest_scores, latest_boxes = np.array([]), np.array([]), np.array([])

for fi in range(total_frames):
    ret, frame_bgr = cap.read()
    if not ret:
        break

    if fi % PROCESS_EVERY == 0:
        frame_pil = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        res = predictor.predict(frame_pil, text_prompt=TEXT_PROMPT, score_threshold=0.15)
        res = nms_results(res, iou_thresh=0.5)
        latest_masks, latest_scores, latest_boxes = res.masks, res.scores, res.boxes

        if fi % 40 == 0:
            print(f"  Frame {fi}/{total_frames}: {len(res.scores)} detections")

    # Draw overlay
    out = frame_bgr.copy()
    for i in range(len(latest_scores)):
        color = COLORS_BGR[i % len(COLORS_BGR)]
        mask = latest_masks[i]
        if mask.shape[0] != H or mask.shape[1] != W:
            mask = cv2.resize(mask.astype(np.float32), (W, H), interpolation=cv2.INTER_NEAREST)
        binary = mask > 0
        for c in range(3):
            out[:, :, c] = np.where(
                binary,
                (out[:, :, c].astype(np.float32) * 0.55 + color[c] * 0.45).astype(np.uint8),
                out[:, :, c],
            )
        contours, _ = cv2.findContours(binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, color, 2)
        x1, y1, x2, y2 = latest_boxes[i].astype(int)
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        label = f"{TEXT_PROMPT} {latest_scores[i]:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(out, (x1, max(y1 - th - 10, 0)), (x1 + tw + 6, max(y1, th + 10)), color, -1)
        cv2.putText(out, label, (x1 + 3, max(y1 - 4, th + 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    writer.write(out)

writer.release()
cap.release()
print(f"\nSaved: {out_path}")

### Preview key frames from the tracking video

In [ ]:
# Show 6 evenly spaced frames from the output video
cap = cv2.VideoCapture(out_path)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
indices = np.linspace(0, total - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(21, 12))
for ax, fi in zip(axes.flatten(), indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ret, frame = cap.read()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {fi}", fontsize=11)
    ax.axis("off")

cap.release()
plt.suptitle(f'SAM3 MLX — Video Tracking "{TEXT_PROMPT}"', fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()